<a href="https://colab.research.google.com/github/ak-ayanat/deep-learning-final-project/blob/master/notebooks/02_preprocessing_baseline_flickr8k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2: Preprocessing + Baseline

Goal: continue after Week 1 EDA and build a first CNN-LSTM image captioning baseline.

Main result: captions are cleaned, vocabulary is built, image features are extracted with a pretrained CNN, and a simple LSTM decoder is trained/evaluated with BLEU score.


## 1. Install and import libraries

This notebook is written for Google Colab. If you run it locally, make sure PyTorch, torchvision, pandas, nltk, and scikit-learn are installed.


In [ ]:
!pip install opendatasets nltk scikit-learn

In [ ]:
import os
import re
import string
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Device: cpu


## 2. Load Flickr8k images and captions

If the dataset is not downloaded yet, run the download cell. It asks for Kaggle username and key.


In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/adityajn105/flickr8k")

text_path = "flickr8k/captions.txt"
images_path = "flickr8k/Images/"

df = pd.read_csv(text_path)
print("Total caption rows:", len(df))
print("Unique images:", df["image"].nunique())
df.head()


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: Ayanat Kanat
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k


100%|██████████| 1.04G/1.04G [00:08<00:00, 135MB/s]



Total caption rows: 40455
Unique images: 8091


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


## 3. Clean captions and add tokens

Captions are lowercased, punctuation is removed, and `<start>` / `<end>` tokens are added.


In [ ]:
def clean_caption(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_caption"] = df["caption"].apply(clean_caption)
df["caption_tokens"] = df["clean_caption"].apply(lambda x: "<start> " + x + " <end>")

df[["image", "caption", "caption_tokens"]].head()


,image,caption,caption_tokens
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...,<start> a child in a pink dress is climbing up...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .,<start> a girl going into a wooden building <end>
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .,<start> a little girl climbing into a wooden p...
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...,<start> a little girl climbing the stairs to h...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...,<start> a little girl in a pink dress going in...


## 4. Split train / validation / test by image

Important: split by unique image names, not by caption rows. This prevents the same image from appearing in different splits.


In [ ]:
unique_images = df["image"].unique()

train_imgs, temp_imgs = train_test_split(
    unique_images, test_size=0.20, random_state=SEED
)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=0.50, random_state=SEED
)

train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

print("Train images:", len(train_imgs), "captions:", len(train_df))
print("Validation images:", len(val_imgs), "captions:", len(val_df))
print("Test images:", len(test_imgs), "captions:", len(test_df))


Train images: 6472 captions: 32360
Validation images: 809 captions: 4045
Test images: 810 captions: 4050


## 5. Build vocabulary

For a baseline, we use words that appear at least `MIN_FREQ` times in the training captions.


In [ ]:
MIN_FREQ = 5

counter = Counter()
for caption in train_df["caption_tokens"]:
    counter.update(caption.split())

special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
words = [word for word, count in counter.items() if count >= MIN_FREQ and word not in special_tokens]

itos = special_tokens + sorted(words)
stoi = {word: idx for idx, word in enumerate(itos)}

PAD_IDX = stoi["<pad>"]
START_IDX = stoi["<start>"]
END_IDX = stoi["<end>"]
UNK_IDX = stoi["<unk>"]

vocab_size = len(itos)
print("Vocabulary size:", vocab_size)

def numericalize(caption):
    return [stoi.get(word, UNK_IDX) for word in caption.split()]

train_df["encoded"] = train_df["caption_tokens"].apply(numericalize)
val_df["encoded"] = val_df["caption_tokens"].apply(numericalize)
test_df["encoded"] = test_df["caption_tokens"].apply(numericalize)

train_df[["caption_tokens", "encoded"]].head()


Vocabulary size: 2655


,caption_tokens,encoded
0,<start> a child in a pink dress is climbing up...,"[1, 11, 443, 1133, 11, 1659, 681, 1161, 473, 2..."
1,<start> a girl going into a wooden building <end>,"[1, 11, 937, 956, 1160, 11, 2625, 322, 2]"
2,<start> a little girl climbing into a wooden p...,"[1, 11, 1306, 937, 473, 1160, 11, 2625, 1686, 2]"
3,<start> a little girl climbing the stairs to h...,"[1, 11, 1306, 937, 473, 2357, 2184, 2397, 1059..."
4,<start> a little girl in a pink dress going in...,"[1, 11, 1306, 937, 1133, 11, 1659, 681, 956, 1..."


## 6. Extract image features using pretrained CNN

We use ResNet-50 pretrained on ImageNet and remove the final classification layer. Each image becomes a 2048-dimensional feature vector.

This cell saves features to disk, so next time it can load them faster.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(device)
feature_extractor.eval()

def extract_features(image_names, batch_size=64, save_path="image_features_resnet50.pt"):
    if os.path.exists(save_path):
        print("Loading saved features:", save_path)
        return torch.load(save_path, map_location="cpu")

    features = {}
    image_names = list(image_names)

    for i in tqdm(range(0, len(image_names), batch_size)):
        batch_names = image_names[i:i + batch_size]
        images = []

        for img_name in batch_names:
            img_path = os.path.join(images_path, img_name)
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
            images.append(image)

        images = torch.stack(images).to(device)

        with torch.no_grad():
            feats = feature_extractor(images)
            feats = feats.squeeze(-1).squeeze(-1).cpu()

        for name, feat in zip(batch_names, feats):
            features[name] = feat

    torch.save(features, save_path)
    return features

all_image_names = df["image"].unique()
image_features = extract_features(all_image_names)
print("Extracted features for images:", len(image_features))
print("Feature size:", next(iter(image_features.values())).shape)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]


  0%|          | 0/127 [00:00<?, ?it/s]

Extracted features for images: 8091
Feature size: torch.Size([2048])


## 7. Dataset and DataLoader

Each row contains one image and one caption. The feature vector is input to the model, and the caption is the target sequence.


In [ ]:
class Flickr8kCaptionDataset(Dataset):
    def __init__(self, dataframe, image_features):
        self.df = dataframe.reset_index(drop=True)
        self.image_features = image_features

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["image"]
        feature = self.image_features[image_name]
        caption = torch.tensor(row["encoded"], dtype=torch.long)
        return feature, caption, image_name

def collate_fn(batch):
    features, captions, image_names = zip(*batch)
    features = torch.stack(features)

    lengths = [len(c) for c in captions]
    max_len = max(lengths)

    padded = torch.full((len(captions), max_len), PAD_IDX, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :len(cap)] = cap

    return features, padded, image_names

BATCH_SIZE = 64

train_loader = DataLoader(
    Flickr8kCaptionDataset(train_df, image_features),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    Flickr8kCaptionDataset(val_df, image_features),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


## 8. Simple CNN-LSTM baseline

The CNN features are already extracted. The model uses:
- Linear layer to project ResNet features
- Embedding layer for words
- LSTM decoder
- Linear layer to predict next word


In [ ]:
class CNNLSTMBaseline(nn.Module):
    def __init__(self, vocab_size, feature_dim=2048, embed_dim=256, hidden_dim=512, pad_idx=0):
        super().__init__()
        self.feature_proj = nn.Linear(feature_dim, embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions):
        # captions input excludes the last token
        captions_in = captions[:, :-1]

        img_embed = self.feature_proj(features).unsqueeze(1)
        word_embed = self.embedding(captions_in)

        # image feature is used as the first input token
        lstm_input = torch.cat([img_embed, word_embed], dim=1)

        output, _ = self.lstm(lstm_input)

        # output predicts all caption tokens including <start> ... <end>
        logits = self.fc(output)
        return logits

model = CNNLSTMBaseline(vocab_size=vocab_size, pad_idx=PAD_IDX).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


CNNLSTMBaseline(
  (feature_proj): Linear(in_features=2048, out_features=256, bias=True)
  (embedding): Embedding(2655, 256, padding_idx=0)
  (lstm): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=2655, bias=True)
)
Trainable parameters: 4143199


## 9. Train baseline

For a real final run, use more epochs. 5 epochs is enough to show that the pipeline works.


In [ ]:
EPOCHS = 5

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for features, captions, _ in tqdm(loader):
        features = features.to(device)
        captions = captions.to(device)

        logits = model(features, captions)

        # targets are the original caption tokens
        targets = captions

        loss = criterion(
            logits.reshape(-1, vocab_size),
            targets.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for features, captions, _ in loader:
            features = features.to(device)
            captions = captions.to(device)

            logits = model(features, captions)
            targets = captions

            loss = criterion(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )
            total_loss += loss.item()

    return total_loss / len(loader)

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss = evaluate_loss(model, val_loader)

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Epoch {epoch}/{EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

torch.save({
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "history": history
}, "cnn_lstm_baseline_week2.pt")

pd.DataFrame(history)


  0%|          | 0/506 [00:00<?, ?it/s]

Epoch 1/5 | train loss: 3.4784 | val loss: 2.9856


  0%|          | 0/506 [00:00<?, ?it/s]

Epoch 2/5 | train loss: 2.7641 | val loss: 2.7050


  0%|          | 0/506 [00:00<?, ?it/s]

...

  0%|          | 0/506 [00:00<?, ?it/s]

Epoch 4/5 | train loss: 2.2874 | val loss: 2.5350


  0%|          | 0/506 [00:00<?, ?it/s]

Epoch 5/5 | train loss: 2.1246 | val loss: 2.5198


,epoch,train_loss,val_loss
0,1,3.478432,2.985552
1,2,2.764129,2.705028
2,3,2.482088,2.588418
3,4,2.287447,2.535026
4,5,2.124637,2.519805


## 10. Generate captions

Greedy decoding is used for the baseline.


In [ ]:
def generate_caption(model, feature, max_len=25):
    model.eval()

    generated = []
    input_word = torch.tensor([[START_IDX]], dtype=torch.long).to(device)
    feature = feature.unsqueeze(0).to(device)

    with torch.no_grad():
        img_embed = model.feature_proj(feature).unsqueeze(1)
        output, hidden = model.lstm(img_embed)

        for _ in range(max_len):
            word_embed = model.embedding(input_word)
            output, hidden = model.lstm(word_embed, hidden)
            logits = model.fc(output.squeeze(1))
            predicted_idx = logits.argmax(dim=1).item()

            if predicted_idx == END_IDX:
                break

            generated.append(itos[predicted_idx])
            input_word = torch.tensor([[predicted_idx]], dtype=torch.long).to(device)

    return " ".join(generated)

# Example prediction from validation set
example_img = val_df.iloc[0]["image"]
print("Image:", example_img)
print("Prediction:", generate_caption(model, image_features[example_img]))

print("\nReal captions:")
for cap in val_df[val_df["image"] == example_img]["clean_caption"].tolist():
    print("-", cap)


Image: 104136873_5b5d41be75.jpg
Prediction: a man is riding a mountain bike down a hill

Real captions:
- people sit on the mountainside and check out the view
- three people are on a hilltop overlooking a green valley
- three people hang out on top of a big hill
- three people overlook a green valley
- three people rest on a ledge above the moutains


## 11. Evaluate with BLEU score

BLEU compares generated captions against the real reference captions for each image.


In [ ]:
def evaluate_bleu(model, eval_df, image_features, max_images=None):
    references = []
    hypotheses = []

    image_names = eval_df["image"].unique()
    if max_images is not None:
        image_names = image_names[:max_images]

    for image_name in tqdm(image_names):
        refs = eval_df[eval_df["image"] == image_name]["clean_caption"].tolist()
        refs = [ref.split() for ref in refs]

        pred = generate_caption(model, image_features[image_name])
        hyp = pred.split()

        references.append(refs)
        hypotheses.append(hyp)

    smoothie = SmoothingFunction().method4

    bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    return {
        "BLEU-1": bleu1,
        "BLEU-2": bleu2,
        "BLEU-4": bleu4
    }

bleu_scores = evaluate_bleu(model, val_df, image_features)
bleu_scores


  0%|          | 0/809 [00:00<?, ?it/s]

{'BLEU-1': 0.5794096121005123,
 'BLEU-2': 0.3905348702064096,
 'BLEU-4': 0.16804005970143443}

## 12. Save Week 2 results

Use the printed BLEU score in `reports/week-02.md`.


In [ ]:
os.makedirs("reports", exist_ok=True)

results_df = pd.DataFrame(history)
results_df.to_csv("reports/week2_training_history.csv", index=False)

with open("reports/week2_bleu_scores.txt", "w") as f:
    for key, value in bleu_scores.items():
        f.write(f"{key}: {value:.4f}\n")

print("Saved:")
print("- reports/week2_training_history.csv")
print("- reports/week2_bleu_scores.txt")


Saved:
- reports/week2_training_history.csv
- reports/week2_bleu_scores.txt
